In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize

from skclust.neighbors import KNeighborsCosineSimilarity
from skclust.neighbors import CosineSimilarityClassifier

# Generate synthetic L2-normalized embeddings
np.random.seed(42)
n_samples = 100
n_features = 64
n_classes = 5

# Create clustered data for classification
X_raw = []
y = []
for class_idx in range(n_classes):
    # Each class has a centroid + noise
    centroid = np.random.randn(n_features)
    samples = centroid + np.random.randn(n_samples // n_classes, n_features) * 0.3
    X_raw.append(samples)
    y.extend([f"class_{class_idx}"] * (n_samples // n_classes))

X = normalize(np.vstack(X_raw), norm='l2').astype(np.float32)
y = np.array(y)

# Create DataFrame version for index testing
df_X = pd.DataFrame(X, index=[f"sample_{i}" for i in range(len(X))])

print("=" * 60)
print("TEST: KNeighborsCosineSimilarity")
print("=" * 60)


# Test with numpy array
knn = KNeighborsCosineSimilarity(n_neighbors=5, backend='auto')
knn.fit(X)

print(f"Backend used: {knn.backend_}")
print(f"Similarities shape: {knn.similarities_.shape}")
print(f"Indices shape: {knn.indices_.shape}")
print(f"First sample's neighbors: {knn.indices_[0]}")
print(f"First sample's similarities: {knn.similarities_[0]}")

# Verify self is first neighbor (similarity ~1.0)
assert np.allclose(knn.similarities_[:, 0], 1.0, atol=1e-5), "Self should be most similar"
assert np.all(knn.indices_[:, 0] == np.arange(len(X))), "Self should be first neighbor"
print("✓ Self-neighbor check passed")

# Test with DataFrame (preserves index)
knn_df = KNeighborsCosineSimilarity(n_neighbors=5, backend='auto')
knn_df.fit(df_X)
assert hasattr(knn_df, 'index_labels_'), "Should preserve DataFrame index"
assert list(knn_df.index_labels_[:3]) == ['sample_0', 'sample_1', 'sample_2']
print("✓ DataFrame index preservation passed")

# Test transform on new data
X_query = normalize(np.random.randn(3, n_features), norm='l2').astype(np.float32)
sims, idxs = knn.transform(X_query)
assert sims.shape == (3, 5), f"Expected (3, 5), got {sims.shape}"
assert idxs.shape == (3, 5), f"Expected (3, 5), got {idxs.shape}"
print("✓ Transform on new data passed")

# Test fit_transform returns cached values
sims_ft, idxs_ft = knn.fit_transform(X)
assert np.array_equal(sims_ft, knn.similarities_), "fit_transform should return cached similarities"
print("✓ fit_transform caching passed")

try:
    import igraph as ig
    
    graph = knn.to_igraph(include_self=False)
    print(f"Graph: {graph.vcount()} vertices, {graph.ecount()} edges")
    assert graph.vcount() == len(X), "Should have n_samples vertices"
    # Each sample contributes (n_neighbors - 1) edges when excluding self
    expected_edges = len(X) * (knn.n_neighbors - 1)
    assert graph.ecount() == expected_edges, f"Expected {expected_edges} edges, got {graph.ecount()}"
    print("✓ igraph conversion passed")
    
    # Test with custom index
    graph_labeled = knn_df.to_igraph(index="auto", include_self=False)
    vertex_names = graph_labeled.vs['name']
    assert 'sample_0' in vertex_names, "Should use DataFrame index as vertex names"
    print("✓ igraph with labels passed")
    
except ImportError:
    print("⚠ igraph not installed, skipping graph tests")

print("\n" + "=" * 60)
print("ALL TESTS PASSED ✓")
print("=" * 60)

TEST: KNeighborsCosineSimilarity
Backend used: faiss
Similarities shape: (100, 5)
Indices shape: (100, 5)
First sample's neighbors: [ 0  4 11  8  7]
First sample's similarities: [1.         0.9378199  0.9229973  0.92078644 0.9153064 ]
✓ Self-neighbor check passed
✓ DataFrame index preservation passed
✓ Transform on new data passed
✓ fit_transform caching passed
Graph: 100 vertices, 400 edges
✓ igraph conversion passed
✓ igraph with labels passed

ALL TESTS PASSED ✓


In [2]:
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize

# Generate synthetic L2-normalized embeddings with varying class separations
np.random.seed(42)
n_samples_per_class = [200, 150, 100, 80, 50]  # Imbalanced classes
n_features = 64
n_classes = len(n_samples_per_class)

X_raw = []
y = []
for class_idx, n_samples in enumerate(n_samples_per_class):
    # Each class has a centroid + noise (varying tightness)
    centroid = np.random.randn(n_features) * 2
    noise_scale = 0.2 + class_idx * 0.1  # Classes get progressively noisier
    samples = centroid + np.random.randn(n_samples, n_features) * noise_scale
    X_raw.append(samples)
    y.extend([f"class_{class_idx}"] * n_samples)

X = normalize(np.vstack(X_raw), norm='l2').astype(np.float32)
y = np.array(y)

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features, {n_classes} classes")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

# Create DataFrame version
df_X = pd.DataFrame(X, index=[f"sample_{i}" for i in range(len(X))])

print("\n" + "=" * 60)
print("TEST: CosineSimilarityClassifier - Basic Functionality")
print("=" * 60)

from skclust.neighbors import CosineSimilarityClassifier

clf = CosineSimilarityClassifier(
    confidence_interval=95,
    backend='auto',
    max_samples_per_class=None,  # Use all
    max_samples_between_classes=5000,
    show_progress=True,
)
clf.fit(X, y)

print(f"\nBackend: {clf.backend_}")
print(f"Classes: {clf.classes_}")

# Check per-class attributes
print("\n--- Per-class cutoffs ---")
for cls in clf.classes_:
    print(f"  {cls}: cutoff={clf.cutoffs_[cls]:.4f}, "
          f"within_ci={clf.within_ci_[cls]}, "
          f"n_within={len(clf.within_class_similarities_[cls])}, "
          f"n_between={len(clf.between_class_similarities_[cls])}")

# Verify each class has its own distribution
assert len(clf.cutoffs_) == n_classes, "Should have cutoff per class"
assert len(clf.within_ci_) == n_classes, "Should have within_ci per class"
assert len(clf.between_ci_) == n_classes, "Should have between_ci per class"
print("✓ Per-class attributes verified")

# Display summary
print("\n--- Summary ---")
print(clf.summary())

print("\n" + "=" * 60)
print("TEST: Prediction with iterative neighbor checking")
print("=" * 60)

# Predict on training data
y_pred = clf.predict(X, k=10)
accuracy = np.mean(y_pred == y)
unknown_rate = np.mean(y_pred == -1)
print(f"Training accuracy (excl. unknown): {accuracy:.2%}")
print(f"Unknown rate: {unknown_rate:.2%}")
assert accuracy > 0.7, f"Expected >70% accuracy, got {accuracy:.2%}"
print("✓ Prediction check passed")

# Predict on new query data
X_query = normalize(np.random.randn(10, n_features), norm='l2').astype(np.float32)
y_query = clf.predict(X_query, k=5)
print(f"Query predictions: {y_query}")

print("\n" + "=" * 60)
print("TEST: search() with different filter_by_cutoff options")
print("=" * 60)

# Test 1: No filtering
results_no_filter = clf.search(X[:3], k=5, filter_by_cutoff=False)
print(f"No filter - first query hits: {len(results_no_filter[0])} neighbors")
assert all(len(hits) == 5 for hits in results_no_filter.values()), "Should return all 5"
print("✓ filter_by_cutoff=False passed")

# Test 2: Use stored cutoffs
results_stored = clf.search(X[:3], k=5, filter_by_cutoff=True)
print(f"Stored cutoffs - first query hits: {len(results_stored[0])} neighbors (filtered)")
# Filtered results should have <= unfiltered
assert len(results_stored[0]) <= len(results_no_filter[0])
print("✓ filter_by_cutoff=True passed")

# Test 3: Scalar cutoff
results_scalar = clf.search(X[:3], k=5, filter_by_cutoff=0.9)
print(f"Scalar cutoff (0.9) - first query hits: {len(results_scalar[0])} neighbors")
# Verify all returned similarities >= 0.9
for hits in results_scalar.values():
    for _, _, sim in hits:
        assert sim >= 0.9, f"Similarity {sim} should be >= 0.9"
print("✓ filter_by_cutoff=<scalar> passed")

# Test 4: Mapping cutoff
custom_cutoffs = {cls: 0.8 for cls in clf.classes_}
custom_cutoffs["class_0"] = 0.95  # Stricter for class_0
results_mapping = clf.search(X[:3], k=5, filter_by_cutoff=custom_cutoffs)
print(f"Mapping cutoffs - first query hits: {len(results_mapping[0])} neighbors")
print("✓ filter_by_cutoff=<Mapping> passed")

# Test 5: pd.Series cutoff
cutoff_series = pd.Series(custom_cutoffs)
results_series = clf.search(X[:3], k=5, filter_by_cutoff=cutoff_series)
print("✓ filter_by_cutoff=<pd.Series> passed")

# Test 6: Missing class in mapping should raise
try:
    bad_cutoffs = {"class_0": 0.5}  # Missing other classes
    clf.search(X[:1], k=5, filter_by_cutoff=bad_cutoffs)
    assert False, "Should have raised ValueError"
except ValueError as e:
    print(f"✓ Missing classes raises ValueError: {e}")

print("\n" + "=" * 60)
print("TEST: DataFrame index preservation")
print("=" * 60)

df_query = df_X.iloc[:3]
results_df = clf.search(df_query, k=3, filter_by_cutoff=False)
assert list(results_df.keys()) == ['sample_0', 'sample_1', 'sample_2']
print(f"Search keys: {list(results_df.keys())}")
print("✓ DataFrame index preserved in search")

print("\n" + "=" * 60)
print("TEST: get_cutoff() method")
print("=" * 60)

all_cutoffs = clf.get_cutoff()
print(f"All cutoffs: {all_cutoffs}")
assert isinstance(all_cutoffs, dict)
assert set(all_cutoffs.keys()) == set(clf.classes_)

single_cutoff = clf.get_cutoff("class_0")
print(f"class_0 cutoff: {single_cutoff}")
assert isinstance(single_cutoff, float)
print("✓ get_cutoff() passed")

print("\n" + "=" * 60)
print("TEST: Edge cases")
print("=" * 60)

# Single sample query
single_result = clf.search(X[:1], k=3)
assert len(single_result) == 1
print("✓ Single sample query passed")

# k larger than dataset
large_k_result = clf.search(X[:1], k=10000)
assert len(large_k_result[0]) <= len(X)
print("✓ Large k handled correctly")

print("\n" + "=" * 60)
print("ALL TESTS PASSED ✓")
print("=" * 60)

Dataset: 580 samples, 64 features, 5 classes
Class distribution: {'class_0': 200, 'class_1': 150, 'class_2': 100, 'class_3': 80, 'class_4': 50}

TEST: CosineSimilarityClassifier - Basic Functionality


Processing classes: 100%|██████████| 5/5 [00:00<00:00, 57.19it/s]


Backend: faiss
Classes: ['class_0' 'class_1' 'class_2' 'class_3' 'class_4']

--- Per-class cutoffs ---
  class_0: cutoff=0.9839, within_ci=(0.9839002609252929, 0.9918711438775063), n_within=19900, n_between=5000
  class_1: cutoff=0.9674, within_ci=(0.9674003601074219, 0.9841390281915665), n_within=11175, n_between=5000
  class_2: cutoff=0.9497, within_ci=(0.9497247397899627, 0.9734963268041611), n_within=4950, n_between=5000
  class_3: cutoff=0.9229, within_ci=(0.9229126065969467, 0.9625864624977112), n_within=3160, n_between=5000
  class_4: cutoff=0.8784, within_ci=(0.8783733010292053, 0.9406106829643248), n_within=1225, n_between=5000
✓ Per-class attributes verified

--- Summary ---
         n_samples  n_within_pairs  n_between_pairs  within_mean  \
class                                                              
class_0        200           19900             5000     0.988213   
class_1        150           11175             5000     0.976764   
class_2        100            495